# Lab 7 — Large Language Models (LLMs)



## The Three Main Transformer Model Families

| Family | Example Models | Main Use |
|---|---|---|
| **Encoder-only** | BERT, RoBERTa | Classification, sentiment analysis, NER |
| **Encoder–decoder** | T5, BART | Translation, summarization, question answering |
| **Decoder-only** | GPT family | Text generation, conversation, completion |


## 1. Setup


In [ ]:
!pip -q install transformers sentencepiece torch


In [ ]:
from transformers import pipeline, set_seed

print("Libraries loaded successfully!")


# Part A — Decoder-Only Model

## Decoder-only example: GPT-style Text Generation

A decoder-only model predicts the next token based on previous tokens.

We will use **DistilGPT-2**, a small GPT-style model.


In [ ]:
generator = pipeline("text-generation", model="distilgpt2")

prompt = "Natural language processing is"

output = generator(prompt, max_new_tokens=30, do_sample=False)

print(output[0]["generated_text"])

# Part B — Encoder–Decoder Model

## Encoder–decoder example: Instruction / Text-to-Text Generation

Encoder–decoder models contain two main parts:

1. The **encoder** reads the input.
2. The **decoder** produces the output.

We will use **FLAN-T5-small**.


In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")

prompt = "Translate to French: I love natural language processing."

# Convert text into tokens
output = model.generate(**tokenizer(prompt, return_tensors="pt"))

# Convert tokens back to text
print(tokenizer.decode(output[0], skip_special_tokens=True))

# Part C — Encoder-Only Model

## Encoder-only example: Sentiment Classification

Encoder-only models read the full input and create a representation of it.

They are very useful when the task is to **understand or classify text**.

Here we use a DistilBERT sentiment model.


In [ ]:
classifier = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")

text = "This NLP lab is easy and interesting."

result = classifier(text)
print(result)

# Tasks

Complete the following tasks.

## Task 1 — Model Families

For each task below, choose the most suitable model family:

1. Sentiment classification
2. Machine translation
3. Open-ended text generation
4. Named Entity Recognition
5. Text summarization

In [ ]:
model_families = {
    "Sentiment classification": "Encoder-only (e.g. BERT, RoBERTa)",
    "Machine translation": "Encoder-decoder (e.g. T5, BART)",
    "Open-ended text generation": "Decoder-only (e.g. GPT)",
    "Named Entity Recognition": "Encoder-only (e.g. BERT, RoBERTa)",
    "Text summarization": "Encoder-decoder (e.g. T5, BART)",
}

for i, (task, family) in enumerate(model_families.items(), 1):
    print(f"{i}. {task} -> {family}")

## Task 2 — Encoder-only Model

Use the sentiment classifier on these three sentences:

```text
The movie was excellent.
The lecture was difficult to understand.
I enjoyed learning about transformers.
```

Print the predicted label and score for each sentence.

In [ ]:
sentences = [
    "The movie was excellent.",
    "The lecture was difficult to understand.",
    "I enjoyed learning about transformers."
]

results = classifier(sentences)

for sentence, result in zip(sentences, results):
    print(f"Sentence: {sentence}")
    print(f"Label: {result['label']} | Score: {result['score']:.4f}\n")

## Task 3 — Encoder–Decoder Model

Use the FLAN-T5 model to perform the following:

- Summarize a short paragraph.


In [ ]:
paragraph = (
    "Natural language processing is a field of artificial intelligence that helps computers "
    "understand, interpret, and generate human language. It is used in many applications such as "
    "machine translation, chatbots, sentiment analysis, and search engines. Recent progress in NLP "
    "comes mainly from transformer models, which are trained on very large amounts of text and can "
    "perform many language tasks with high accuracy."
)

prompt = "Summarize: " + paragraph

inputs = tokenizer(prompt, return_tensors="pt")
output = model.generate(**inputs, max_new_tokens=50, num_beams=4)

print("Original paragraph:\n", paragraph)
print("\nSummary:\n", tokenizer.decode(output[0], skip_special_tokens=True))


## Task 5 — Short Reflection

Answer in 3–5 sentences:

**Why is a decoder-only model suitable for a chatbot such as ChatGPT?**


**Answer:**

A decoder-only model is trained to predict the next token given all the previous tokens, which is exactly what a chatbot does: it reads the conversation so far and generates the reply one token at a time. Its causal (masked) self-attention lets it use the whole conversation history as context, so the prompt and the answer are handled as one continuous sequence. This makes it naturally good at open-ended, free-form text generation of any length. It also scales very well when trained on huge amounts of text, and after instruction tuning and RLHF it can follow instructions and hold multi-turn conversations like ChatGPT.

# Optional Challenge

Try generating text several times with the same prompt.

Remove:

```python
set_seed(42)
```

and run the sampling cell multiple times.

### Question

Why do you sometimes get different outputs even when the prompt is the same?

**Hint:** Sampling introduces randomness when choosing the next token.


In [ ]:
# With a fixed seed -> same output every run
set_seed(42)
prompt = "Natural language processing is"

for i in range(3):
    out = generator(prompt, max_new_tokens=30, do_sample=True, temperature=0.9, top_k=50)
    print(f"Run {i+1}: {out[0]['generated_text']}\n")

In [ ]:
# Without set_seed -> different output every run
prompt = "Natural language processing is"

for i in range(3):
    out = generator(prompt, max_new_tokens=30, do_sample=True, temperature=0.9, top_k=50)
    print(f"Run {i+1}: {out[0]['generated_text']}\n")

**Answer:**

With `do_sample=True`, the model does not always pick the most likely next token. It samples the next token randomly from the probability distribution (controlled by `temperature` and `top_k`), so each run can choose different tokens and the whole continuation changes. `set_seed(42)` fixes the random number generator, so the same random choices are made and the output is reproducible. With `do_sample=False` (greedy decoding), the output is always the same because the highest-probability token is always chosen.